# Uniform-cuboid FMM versus exact dense cuboid direct

This notebook compares the complete `UNIFORM_CUBOID -> POINT` FMM with the validated exact `DenseDirectPlan`. Targets coincide with cube centres, so every finite self field is included. The far field uses analytical cuboid P2M while list1 uses the same exact pair tensor as dense direct.


In [ ]:
import time
import cdfmm
import matplotlib.pyplot as plt
import numpy as np

SIDE = 10.0e-9
SPACING = 3.0 * SIDE
SHAPE = (8, 8, 8)
ORDERS = [1, 2, 3, 4, 5, 6, 8]
REPEATS = 5

indices = np.indices(SHAPE, dtype=float).reshape(3, -1).T
centres = (indices - (np.asarray(SHAPE) - 1) / 2) * SPACING
phase = np.arange(len(centres), dtype=float)
magnetisations = np.column_stack((
    7.0e5 + 1.5e5 * np.sin(0.17 * phase),
    -3.0e5 + 2.0e5 * np.cos(0.11 * phase),
    4.0e5 * np.sin(0.07 * phase + 0.3),
))
cube = cdfmm.CuboidSize(SIDE, SIDE, SIDE)
moments = SIDE**3 * magnetisations


## Exact reusable reference

Construction and repeated evaluation are timed separately. No MagTense dependency is involved.


In [ ]:
start = time.perf_counter()
direct = cdfmm.DenseDirectPlan(
    source_positions=centres,
    target_positions=centres,
    source_geometry=cdfmm.SourceGeometry.UNIFORM_CUBOID,
    target_geometry=cdfmm.TargetGeometry.POINT,
    source_sizes=[cube],
)
direct_initialisation_s = time.perf_counter() - start
direct.evaluate(moments, backend=cdfmm.DenseDirectBackend.PORTABLE)
samples = []
for _ in range(REPEATS):
    start = time.perf_counter()
    H_direct = direct.evaluate(moments, backend=cdfmm.DenseDirectBackend.PORTABLE)
    samples.append(time.perf_counter() - start)
direct_evaluation_s = np.median(samples)
print(f"Dense construction: {direct_initialisation_s:.6f} s")
print(f"Dense evaluation:   {direct_evaluation_s:.6f} s")


## Expansion-order convergence and timing

The meaningful pointwise denominator excludes fields smaller than $10^{-12}$ of the maximum reference magnitude. Order five is included because cube symmetry makes it the first degree carrying the physical finite-size correction to the external point-dipole field.


In [ ]:
rows = []
reference_norms = np.linalg.norm(H_direct, axis=1)
meaningful = reference_norms > 1.0e-12 * reference_norms.max()
for order in ORDERS:
    options = cdfmm.UniformFmmOptions()
    options.expansion_order = order
    options.tree.max_level = 3
    options.source_geometry = cdfmm.SourceGeometry.UNIFORM_CUBOID
    options.source_sizes = [cube]
    options.backend = cdfmm.ExecutionBackend.CPU_STATIC
    start = time.perf_counter()
    fmm = cdfmm.UniformFmm(centres, centres, options)
    initialisation_s = time.perf_counter() - start
    fmm.evaluate(moments)
    samples = []
    for _ in range(REPEATS):
        start = time.perf_counter()
        H_fmm = fmm.evaluate(moments)["H"]
        samples.append(time.perf_counter() - start)
    difference = H_fmm - H_direct
    absolute = np.linalg.norm(difference, axis=1)
    rows.append(dict(
        order=order,
        initialisation_s=initialisation_s,
        evaluation_s=float(np.median(samples)),
        relative_l2=float(np.linalg.norm(difference) / np.linalg.norm(H_direct)),
        maximum_absolute=float(absolute.max()),
        maximum_pointwise_relative=float(
            np.max(absolute[meaningful] / reference_norms[meaningful])
        ),
    ))

for row in rows:
    print(row)


In [ ]:
orders = [row["order"] for row in rows]
errors = [row["relative_l2"] for row in rows]
times = [row["evaluation_s"] for row in rows]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].semilogy(orders, errors, "o-")
axes[0].set(xlabel="Expansion order p", ylabel="Relative L2 error",
            title="Cuboid FMM convergence")
axes[1].plot(orders, times, "o-")
axes[1].set(xlabel="Expansion order p", ylabel="Evaluation time [s]",
            title="Repeated FMM evaluation")
for axis in axes:
    axis.grid(alpha=0.3)
fig.tight_layout()
plt.show()


## Scope

This validates axis-aligned uniformly magnetised cuboid sources and point targets on the CPU static backend. It does not introduce point-to-cuboid, cuboid-averaged targets, rotated cuboids, or target-volume L2P.
